In [22]:
import os
import json
import pandas as pd

BASE_DIR = r"G:\jupyter\2025APMCM"
TARIFF_DIR = os.path.join(BASE_DIR, "Tariff Data")
JSON_FILE = os.path.join(BASE_DIR, "tariff_columns_by_year.json")

if os.path.exists(JSON_FILE):
    with open(JSON_FILE, 'r') as f:
        cols_meta = json.load(f)

years = range(2015, 2026)
available_files = {}
for year in years:
    folder_path = os.path.join(TARIFF_DIR, f"tariff_data_{year}")
    
    file_txt = os.path.join(folder_path, f"tariff_database_{year}.txt")
    file_xlsx = os.path.join(folder_path, f"tariff_database_{year}.xlsx")
    file_csv = os.path.join(folder_path, f"tariff_database_{year}.csv")
    
    found_path = None
    if os.path.exists(file_txt):
        found_path = file_txt
    elif os.path.exists(file_xlsx):
        found_path = file_xlsx
    elif os.path.exists(file_csv):
        found_path = file_csv
    
    if found_path:
        available_files[year] = found_path

Working Directory: G:\jupyter\2025APMCM
[OK] JSON Metadata loaded. Keys: ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']

--- File Inventory Check ---
[OK] 2015: Found TXT
[OK] 2016: Found TXT
[OK] 2017: Found TXT
[OK] 2018: Found TXT
[OK] 2019: Found XLSX
[OK] 2020: Found XLSX
[OK] 2021: Found XLSX
[OK] 2022: Found XLSX
[OK] 2023: Found XLSX
[OK] 2024: Found XLSX
[OK] 2025: Found TXT

--- Sample Reading (Head 3 rows) ---

Reading 2015 from tariff_database_2015.txt...
Raw Header Sample: hts8|brief_description|quantity_1_code|quantity_2_code|wto_binding_code|mfn_text_rate|mfn_rate_type_...
Columns detected: ['hts8|brief_description|qu', 'ntity_1_code|qu', 'ntity_2_code|wto_binding_code|mfn_text_r', 'te|mfn_r', 'te_type_code|mfn_'] ...
Shape: (3, 257)

Reading 2025 from tariff_database_2025.txt...
Raw Header Sample: hts8,brief_description,quantity_1_code,quantity_2_code,wto_binding_code,mfn_text_rate,mfn_rate_type_...
Columns detected: ['hts8', 'b

In [20]:
import pandas as pd
import os
import numpy as np

BASE_DIR = r"G:\jupyter\2025APMCM"
TARIFF_DIR = os.path.join(BASE_DIR, "Tariff Data")

def read_tariff_year(year):
    folder_path = os.path.join(TARIFF_DIR, f"tariff_data_{year}")
    file_txt = os.path.join(folder_path, f"tariff_database_{year}.txt")
    file_xlsx = os.path.join(folder_path, f"tariff_database_{year}.xlsx")
    
    df = None
    if os.path.exists(file_xlsx):
        df = pd.read_excel(file_xlsx, dtype={'hts8': str})
    elif os.path.exists(file_txt):
        sep = ',' if year == 2025 else '|'
        df = pd.read_csv(file_txt, sep=sep, encoding='latin1', 
                         dtype={'hts8': str}, on_bad_lines='skip')
    
    if df is None:
        return None

    df.columns = [c.lower().strip() for c in df.columns]
    if 'hts8' in df.columns:
        df['hts8'] = df['hts8'].astype(str).str.replace('.', '', regex=False).str.strip()
        df = df[df['hts8'].str.len() >= 4]
    return df

def clean_rate_column(series):
    s = series.astype(str).str.lower().str.strip()
    s = s.replace({'free': '0', 'nan': '0', 'nm': '0'})
    s = s.str.replace('%', '', regex=False)
    vals = pd.to_numeric(s, errors='coerce').fillna(0.0)
    return vals

# Perform the analysis silently
test_years = [2015, 2020, 2025]
summary_stats_list = []
for y in test_years:
    df_temp = read_tariff_year(y)
    if df_temp is not None and 'mfn_ad_val_rate' in df_temp.columns:
        df_temp['rate_clean'] = clean_rate_column(df_temp['mfn_ad_val_rate'])
        summary_stats_list.append({
            'Year': y,
            'Mean_Rate': df_temp['rate_clean'].mean(),
            'Max_Rate': df_temp['rate_clean'].max(),
            'Zero_Tariff_Pct': (df_temp['rate_clean'] == 0).mean()
        })
summary_stats = pd.DataFrame(summary_stats_list)

[2015] Loading XLSX...
[2020] Loading XLSX...
[2025] Loading XLSX...

--- Tariff Rate Analysis (MFN) ---
   Year                     Raw_Sample   Mean_Rate     Max_Rate  \
0  2015  [0.0, 0.0, 0.068, 0.0, 0.045]  483.703303  9999.999999   
1  2020  [0.0, 0.0, 0.068, 0.068, 0.0]  377.081712  9999.999999   
2  2025  [0.0, 0.0, 0.068, 0.0, 0.045]  505.714241  9999.999999   

   Zero_Tariff_Pct  
0         0.477544  
1         0.375033  
2         0.467848  


In [24]:
import pandas as pd
import os
import numpy as np

BASE_DIR = r"G:\jupyter\2025APMCM"
TARIFF_DIR = os.path.join(BASE_DIR, "Tariff Data")

def analyze_rate_distribution(year):
    folder_path = os.path.join(TARIFF_DIR, f"tariff_data_{year}")
    file_xlsx = os.path.join(folder_path, f"tariff_database_{year}.xlsx")
    file_txt = os.path.join(folder_path, f"tariff_database_{year}.txt")
    
    df = None
    if os.path.exists(file_xlsx):
        df = pd.read_excel(file_xlsx, dtype={'hts8': str})
    elif os.path.exists(file_txt):
        sep = ',' if year == 2025 else '|'
        df = pd.read_csv(file_txt, sep=sep, encoding='latin1', dtype={'hts8': str}, on_bad_lines='skip')
    
    if df is None or 'mfn_ad_val_rate' not in df.columns: 
        return None

    df.columns = [c.lower().strip() for c in df.columns]
    s = df['mfn_ad_val_rate'].astype(str).str.replace('%', '', regex=False)
    s = s.replace({'free': '0', 'nan': '0', 'nm': '0'})
    vals = pd.to_numeric(s, errors='coerce').fillna(0.0)
    
    clean_vals = vals[vals < 10.0]
    return {'Year': year, 'Clean_Mean': clean_vals.mean()}

years_to_check = [2015, 2018, 2020, 2024, 2025]
results = [analyze_rate_distribution(y) for y in years_to_check]
stats_df = pd.DataFrame([res for res in results if res is not None])
mean_2025 = stats_df[stats_df['Year'] == 2025]['Clean_Mean'].values[0]

# The logic of checking `mean_2025 < 0.10` is implicitly executed.
# The core outcome is that the data is confirmed to be pre-shock.

--- Deep Cleaning Analysis ---
Processing 2015...
Processing 2018...
Processing 2020...
Processing 2024...
Processing 2025...

    Year  Clean_Mean  Clean_Median  Noise_Count (>=10)  Valid_Count
0  2015    0.036739         0.000                 616        12120
1  2018    0.037585         0.012                 622        12578
2  2020    0.044436         0.030                 846        21592
3  2024    0.037490         0.006                 455        12520
4  2025    0.037823         0.011                 659        12373

[CONCLUSION] 2025 Clean Mean is 0.0378 (3.78%).
This confirms the data is PRE-SHOCK. We must model the Trump tariffs manually.


In [30]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = r"G:\jupyter\2025APMCM"
TARIFF_DIR = os.path.join(BASE_DIR, "Tariff Data")
OUTPUT_DIR = os.path.join(BASE_DIR, "Cleaned_Data")
SAVE_PATH = os.path.join(OUTPUT_DIR, "Figure_1_Tariff_Baseline.png")
os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.rcParams.update({
    'font.family': 'serif', 'font.serif': ['Times New Roman'], 
    'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 12
})

def get_clean_rates(year):
    folder_path = os.path.join(TARIFF_DIR, f"tariff_data_{year}")
    file_xlsx = os.path.join(folder_path, f"tariff_database_{year}.xlsx")
    file_txt = os.path.join(folder_path, f"tariff_database_{year}.txt")
    df = None
    if os.path.exists(file_xlsx):
        df = pd.read_excel(file_xlsx, dtype={'hts8': str})
    elif os.path.exists(file_txt):
        sep = ',' if year == 2025 else '|'
        df = pd.read_csv(file_txt, sep=sep, encoding='latin1', dtype={'hts8': str}, on_bad_lines='skip')
    
    if df is not None and 'mfn_ad_val_rate' in df.columns:
        df.columns = [c.lower().strip() for c in df.columns]
        s = df['mfn_ad_val_rate'].astype(str).str.replace('%', '', regex=False)
        s = s.replace({'free': '0', 'nan': '0', 'nm': '0'})
        vals = pd.to_numeric(s, errors='coerce').fillna(0.0)
        return vals[vals < 10.0]
    return pd.Series(dtype=float)

years = range(2015, 2026)
yearly_means = []
data_2025 = None
for y in years:
    rates = get_clean_rates(y)
    if not rates.empty:
        yearly_means.append({'Year': y, 'Mean_Rate': rates.mean() * 100})
        if y == 2025:
            data_2025 = rates * 100
ts_df = pd.DataFrame(yearly_means)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.lineplot(data=ts_df, x='Year', y='Mean_Rate', marker='o', markersize=8, ax=axes[0], linewidth=2.5, color='#2b579a')
axes[0].set_title('(a) Historical MFN Tariff Rate Trend (2015-2025)', fontsize=14, fontweight='bold', pad=15)
axes[0].set_ylabel('Average Ad Valorem Rate (%)', fontsize=12)
axes[0].set_ylim(3.0, 5.0)
axes[0].grid(True, which='both', linestyle='--', linewidth=0.5)
axes[0].annotate('Proposed Policy Shock\n(Target: ~20%)', 
                 xy=(2025, 3.8), xytext=(2020, 4.5),
                 arrowprops=dict(facecolor='red', shrink=0.05),
                 fontsize=12, color='#c44e52', fontweight='bold',
                 bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#c44e52", alpha=0.8))

if data_2025 is not None:
    sns.histplot(data_2025, bins=60, ax=axes[1], color='#c44e52', stat='probability', kde=False)
    axes[1].set_title('(b) Distribution of Baseline Tariffs in 2025', fontsize=14, fontweight='bold', pad=15)
    axes[1].set_xlabel('Tariff Rate (%)', fontsize=12)
    axes[1].set_xlim(0, 15)
    axes[1].axvline(x=10, color='k', linestyle='--', linewidth=2, label='Proposed Min Baseline (10%)')
    axes[1].legend(loc='upper right')

plt.tight_layout()
plt.savefig(SAVE_PATH, dpi=300)
plt.close()

Generating corrected plot...

[SUCCESS] Fixed Image saved to: G:\jupyter\2025APMCM\Cleaned_Data\Figure_1_Tariff_Baseline_Fixed.png


In [36]:
import pandas as pd
import os

# --- Configuration ---
BASE_DIR = r"G:\jupyter\2025APMCM"
FILES_TO_INSPECT = [
    os.path.join(BASE_DIR, "Cleaned_Data", "Sector_Base_Tariffs.csv"),
    os.path.join(BASE_DIR, "Trade Data", "US_Import_Cars_Chips_2015_2024.csv"),
    os.path.join(BASE_DIR, "Trade Data", "Soybean_Competition_Exports_to_China_2015_2024.csv")
]

print("--- DATA RECONNAISSANCE REPORT ---\n")

for fpath in FILES_TO_INSPECT:
    fname = os.path.basename(fpath)
    print(f"TARGET: {fname}")
    
    if os.path.exists(fpath):
        try:
            # Attempt read
            df = pd.read_csv(fpath)
            
            # Report
            print(f"STATUS: Found")
            print(f"SHAPE: {df.shape}")
            print(f"COLUMNS: {df.columns.tolist()}")
            print("HEAD (3 rows):")
            print(df.head(3))
            
        except Exception as e:
            print(f"[ERROR] Reading failed: {e}")
    else:
        print(f"[MISSING] File not found at: {fpath}")
    
    print("-" * 60 + "\n")

--- DATA RECONNAISSANCE REPORT ---

TARGET: Sector_Base_Tariffs.csv
STATUS: Found
SHAPE: (308, 4)
COLUMNS: ['Year', 'Sector', 'hts8', 'Base_Rate']
HEAD (3 rows):
   Year   Sector      hts8  Base_Rate
0  2015  Soybean  12019000        0.0
1  2015    Chips  85411000        0.0
2  2015    Chips  85412100        0.0
------------------------------------------------------------

TARGET: US_Import_Cars_Chips_2015_2024.csv
STATUS: Found
SHAPE: (120, 47)
COLUMNS: ['typeCode', 'freqCode', 'refPeriodId', 'refYear', 'refMonth', 'period', 'reporterCode', 'reporterISO', 'reporterDesc', 'flowCode', 'flowDesc', 'partnerCode', 'partnerISO', 'partnerDesc', 'partner2Code', 'partner2ISO', 'partner2Desc', 'classificationCode', 'classificationSearchCode', 'isOriginalClassification', 'cmdCode', 'cmdDesc', 'aggrLevel', 'isLeaf', 'customsCode', 'customsDesc', 'mosCode', 'motCode', 'motDesc', 'qtyUnitCode', 'qtyUnitAbbr', 'qty', 'isQtyEstimated', 'altQtyUnitCode', 'altQtyUnitAbbr', 'altQty', 'isAltQtyEstimated'

In [2]:
import pandas as pd
import os
import numpy as np

# --- 配置 ---
BASE_DIR = r"G:\jupyter\2025APMCM"
TARIFF_DIR = os.path.join(BASE_DIR, "Tariff Data")
TRADE_DIR = os.path.join(BASE_DIR, "Trade Data")
OUTPUT_DIR = os.path.join(BASE_DIR, "Cleaned_Data")
os.makedirs(OUTPUT_DIR, exist_ok=True)

SECTORS = {
    'Soybean': ['1201', '120190'], # 尝试匹配4位或6位
    'Auto': ['8703'],
    'Chips': ['8541', '8542']
}

def clean_hts(val):
    """将HTS转为纯数字字符串"""
    return str(val).replace('.', '').strip()

def get_trade_weights():
    """读取贸易数据并聚合到 4位 HTS 级别，以便匹配"""
    print("Loading Trade Data...")
    files = [
        "US_Import_Cars_Chips_2015_2024.csv",
        "Soybean_Competition_Exports_to_China_2015_2024.csv"
    ]
    dfs = []
    for f in files:
        p = os.path.join(TRADE_DIR, f)
        if os.path.exists(p):
            try:
                # 读取关键列：refYear, cmdCode (HTS), primaryValue
                d = pd.read_csv(p, usecols=['refYear', 'cmdCode', 'primaryValue'])
                d.columns = ['Year', 'hts', 'Value']
                d['hts'] = d['hts'].apply(clean_hts)
                # 截取前4位作为匹配键 (解决 8703 vs 87032300 的问题)
                d['hts_4'] = d['hts'].str.slice(0, 4)
                dfs.append(d)
            except Exception as e:
                print(f"Err reading {f}: {e}")
    
    if dfs:
        full = pd.concat(dfs)
        # 按年和4位码聚合权重
        return full.groupby(['Year', 'hts_4'])['Value'].sum().reset_index()
    return pd.DataFrame()

def process_year(year, trade_weights):
    """处理单年数据"""
    # 读取关税 (复用之前的逻辑，简化版)
    folder = os.path.join(TARIFF_DIR, f"tariff_data_{year}")
    f_xlsx = os.path.join(folder, f"tariff_database_{year}.xlsx")
    f_txt = os.path.join(folder, f"tariff_database_{year}.txt")
    
    df = None
    if os.path.exists(f_xlsx):
        df = pd.read_excel(f_xlsx, dtype={'hts8': str})
    elif os.path.exists(f_txt):
        sep = ',' if year == 2025 else '|'
        df = pd.read_csv(f_txt, sep=sep, encoding='latin1', dtype={'hts8': str}, on_bad_lines='skip')
    
    if df is None: return None
    
    df.columns = [c.lower().strip() for c in df.columns]
    if 'mfn_ad_val_rate' not in df.columns: return None
    
    # 清洗
    df['hts8'] = df['hts8'].apply(clean_hts)
    df['hts_4'] = df['hts8'].str.slice(0, 4) # 生成4位匹配键
    
    s = df['mfn_ad_val_rate'].astype(str).str.replace('%', '').replace({'free': '0', 'nan': '0', 'nm': '0'})
    df['Rate'] = pd.to_numeric(s, errors='coerce').fillna(0.0)
    df = df[df['Rate'] < 10.0] # 过滤异常值
    
    # --- 计算指数 ---
    res = {'Year': year}
    
    # 1. General Index (算术平均，因为我们没有全口径贸易数据)
    res['General_Tau'] = df['Rate'].mean()
    
    # 2. Sector Indices (尝试加权)
    w_year = 2024 if year == 2025 else year
    t_w = trade_weights[trade_weights['Year'] == w_year]
    
    # 将关税表按4位码聚合平均
    tariff_agg = df.groupby('hts_4')['Rate'].mean().reset_index()
    
    # 合并
    merged = pd.merge(tariff_agg, t_w, on='hts_4', how='inner')
    
    for sec, prefixes in SECTORS.items():
        # 匹配前缀
        p_list = [p[:4] for p in prefixes] # 确保前缀也是4位
        sec_data = merged[merged['hts_4'].isin(p_list)]
        
        if not sec_data.empty and sec_data['Value'].sum() > 0:
            # 加权平均
            w_rate = (sec_data['Rate'] * sec_data['Value']).sum() / sec_data['Value'].sum()
            res[f'{sec}_Tau'] = w_rate
        else:
            # 回退：仅用关税数据的算术平均
            raw_match = df[df['hts_4'].isin(p_list)]
            res[f'{sec}_Tau'] = raw_match['Rate'].mean() if not raw_match.empty else 0.0
            
    return res

# --- 主程序 ---
weights = get_trade_weights()
results = []

print("Starting Robust ETL...")
for y in range(2015, 2026):
    try:
        r = process_year(y, weights)
        if r:
            # 模拟冲击列
            r['Simulated_Shock'] = max(r['General_Tau'], 0.10) if y == 2025 else r['General_Tau']
            results.append(r)
            print(f"Processed {y}")
    except Exception as e:
        print(f"Failed {y}: {e}")

final_df = pd.DataFrame(results)
final_df.to_csv(os.path.join(OUTPUT_DIR, "Annual_Tariff_Index.csv"), index=False)
print("Done.")

Loading Trade Data...
Starting Robust ETL...
Processed 2015
Processed 2016
Processed 2017
Processed 2018
Processed 2019
Processed 2020
Processed 2021
Processed 2022
Processed 2023
Processed 2024
Processed 2025
Done.
